# Notebook 06 — Data Curation, Cleaning, and Splitting

Good finetuning runs start with better data, not more optimizer tricks. This notebook builds a transparent curation pipeline for:

- cleaning noisy instruction data
- deduplicating exact and near duplicates
- checking leakage against held-out prompts
- shaping a lightweight curriculum
- balancing task mixtures
- exporting clean JSONL and CSV artifacts

The framing stays Colab Pro friendly and open-source only: we use the same tokenizer and notebook workflow that later Unsloth training runs will consume.

## What you will build

We will start from a noisy local dataset and move through a realistic curation flow:

1. profile quality problems
2. normalize structure and text
3. remove duplicates
4. flag train/eval leakage
5. shape a simple curriculum
6. balance task categories
7. design a validation split
8. export reusable artifacts

## Runtime framing

 Even though this notebook focuses on curation rather than training, it still uses the same open-source stack as the rest of the course so token counts, chat records, and exported files line up with later Unsloth SFT runs.

In [ ]:
# --- Google Colab Pro Fine-Tuning + Evaluation Setup ---
%%capture
!pip install -q --upgrade pip
!pip install -q unsloth datasets trl peft accelerate bitsandbytes pandas matplotlib scikit-learn

import json
import math
import random
import time
from pathlib import Path
from typing import Any, Dict, List

import torch
from datasets import Dataset, DatasetDict, load_dataset
from google.colab import drive
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel

drive.mount('/content/drive')

CACHE_DIR = "/content/drive/MyDrive/models"
OUTPUT_DIR = "/content/drive/MyDrive/castalia-finetuning"
MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048
DTYPE = None
LOAD_IN_4BIT = True

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
    cache_dir=CACHE_DIR,
)

tokenizer.padding_side = "right"

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

def formatting_prompts_func(batch):
    return {"text": batch["text"]}

print("✓ Fine-tuning stack ready")
print("  Model:", MODEL_NAME)
print("  CUDA device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("  BF16 supported:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)

## Step 1: Add helpers and artifact paths

The helpers stay simple on purpose so you can reuse them in your own project data pipelines.

In [ ]:
import csv
import re
from collections import Counter

random.seed(6)

ARTIFACT_DIR = Path(OUTPUT_DIR) / "notebook_06_data_curation"
EXPORT_DIR = ARTIFACT_DIR / "exports"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

def normalize_text(text):
    text = text or ""
    text = re.sub(r"<[^>]+>", " ", text)
    text = text.replace("\u00a0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def fingerprint(prompt, response):
    return " || ".join([normalize_text(prompt).lower(), normalize_text(response).lower()])

def token_estimate(text):
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])

def jaccard_similarity(a, b):
    a_tokens = set(normalize_text(a).lower().split())
    b_tokens = set(normalize_text(b).lower().split())
    if not a_tokens or not b_tokens:
        return 0.0
    return len(a_tokens & b_tokens) / len(a_tokens | b_tokens)

print("Artifacts:", ARTIFACT_DIR.resolve())

## Step 2: Create a noisy raw dataset

This miniature dataset includes several realistic problems:

- exact duplicates
- near duplicates
- inconsistent whitespace
- HTML fragments
- empty responses
- label imbalance
- examples that are too close to evaluation prompts

In [ ]:
raw_records = [
    {"id": "001", "task_type": "qa", "difficulty": "easy", "prompt": "Explain LoRA in one paragraph.", "response": "LoRA trains small low-rank adapter matrices instead of updating the entire base model.", "source": "handwritten"},
    {"id": "002", "task_type": "qa", "difficulty": "easy", "prompt": "Explain LoRA in one paragraph.", "response": "LoRA trains small low-rank adapter matrices instead of updating the entire base model.", "source": "duplicate_exact"},
    {"id": "003", "task_type": "qa", "difficulty": "easy", "prompt": " Explain   LoRA in one paragraph. ", "response": "LoRA trains small low-rank adapter matrices instead of updating the full model. ", "source": "duplicate_near"},
    {"id": "004", "task_type": "comparison", "difficulty": "medium", "prompt": "When is RAG better than finetuning?", "response": "RAG is better when the main issue is missing or changing knowledge.", "source": "handwritten"},
    {"id": "005", "task_type": "comparison", "difficulty": "medium", "prompt": "When is RAG better than finetuning?", "response": "<p>RAG is better when the main issue is missing or changing knowledge.</p>", "source": "html_noise"},
    {"id": "006", "task_type": "checklist", "difficulty": "medium", "prompt": "Give a validation checklist for a first QLoRA run.", "response": "Keep a held-out split, inspect loss curves, compare base vs tuned prompts, and save artifacts.", "source": "handwritten"},
    {"id": "007", "task_type": "qa", "difficulty": "hard", "prompt": "Why can a learning rate be too high for adapter training?", "response": "A learning rate that is too high can destabilize training and create brittle updates.", "source": "handwritten"},
    {"id": "008", "task_type": "qa", "difficulty": "hard", "prompt": "Why can a learning rate be too high for adapter training?", "response": "", "source": "empty_response"},
    {"id": "009", "task_type": "qa", "difficulty": "medium", "prompt": "What makes a good held-out prompt?", "response": "A good held-out prompt is realistic, scorable, and not copied from training data.", "source": "eval_risk"},
    {"id": "010", "task_type": "qa", "difficulty": "medium", "prompt": "What makes a useful held-out prompt for notebook-style evaluation?", "response": "It should be realistic, specific enough to score, and different from the training set.", "source": "eval_risk_near"},
    {"id": "011", "task_type": "checklist", "difficulty": "easy", "prompt": "List three signs of overfitting in SFT.", "response": "Rising validation loss, copied phrasing, and weaker held-out behavior are three common signs.", "source": "handwritten"},
    {"id": "012", "task_type": "checklist", "difficulty": "easy", "prompt": "List 3 signs of overfitting in SFT.", "response": "Rising validation loss, copied phrasing, and weaker held-out behavior are three common signs.", "source": "duplicate_near"},
    {"id": "013", "task_type": "rewrite", "difficulty": "medium", "prompt": "Rewrite this answer to be shorter: Adapter tuning is often cheaper than full finetuning because only a subset of weights changes.", "response": "Adapter tuning is cheaper because only a small subset of weights changes.", "source": "handwritten"},
    {"id": "014", "task_type": "rewrite", "difficulty": "medium", "prompt": "Rewrite this answer to be shorter: Adapter tuning is often cheaper than full finetuning because only a subset of weights changes.", "response": "Adapter tuning is cheaper because only a small subset of weights changes.", "source": "duplicate_exact"},
    {"id": "015", "task_type": "qa", "difficulty": "easy", "prompt": "What does assistant-only loss mean?", "response": "It trains the model on assistant targets rather than penalizing user turns.", "source": "handwritten"},
    {"id": "016", "task_type": "qa", "difficulty": "medium", "prompt": "What is sequence packing?", "response": "Sequence packing combines short examples to reduce wasted padding tokens during training.", "source": "handwritten"},
    {"id": "017", "task_type": "qa", "difficulty": "medium", "prompt": "What is sequence packing?", "response": "Sequence packing combines short examples to reduce wasted padding tokens during training.", "source": "duplicate_exact"},
    {"id": "018", "task_type": "comparison", "difficulty": "hard", "prompt": "Compare keeping an adapter separate vs merging it.", "response": "Separate adapters are easier to version and roll back; merged checkpoints are simpler to deploy as a single artifact.", "source": "handwritten"},
    {"id": "019", "task_type": "comparison", "difficulty": "hard", "prompt": "Compare keeping an adapter separate versus merging it.", "response": "Separate adapters are easier to version and roll back, while merged checkpoints are simpler to hand off.", "source": "duplicate_near"},
    {"id": "020", "task_type": "qa", "difficulty": "hard", "prompt": "Why inspect validation loss and generations together?", "response": "Loss measures optimization, but generations reveal whether behavior actually improved.", "source": "handwritten"},
]

raw_df = pd.DataFrame(raw_records)
raw_df["prompt_tokens"] = raw_df["prompt"].map(token_estimate)
raw_df["response_tokens"] = raw_df["response"].map(token_estimate)

display(raw_df.head())
print("Raw rows:", len(raw_df))

## Step 3: Audit the raw dataset

Before cleaning anything, measure the problems. Otherwise you cannot tell whether the pipeline helped.

In [ ]:
raw_issue_rows = [
    {"issue": "empty_response", "rows": int((raw_df["response"].str.strip() == "").sum())},
    {"issue": "exact_prompt_response_duplicates", "rows": int(raw_df.duplicated(subset=["prompt", "response"]).sum())},
    {"issue": "task_types", "rows": raw_df["task_type"].nunique()},
    {"issue": "mean_prompt_tokens", "rows": round(raw_df["prompt_tokens"].mean(), 1)},
    {"issue": "mean_response_tokens", "rows": round(raw_df["response_tokens"].mean(), 1)},
]
display(pd.DataFrame(raw_issue_rows))

display(raw_df["task_type"].value_counts().rename_axis("task_type").reset_index(name="rows"))

## Step 4: Normalize text and structure

We strip HTML, collapse whitespace, and create stable fingerprints for downstream deduplication.

In [ ]:
clean_df = raw_df.copy()
clean_df["prompt_clean"] = clean_df["prompt"].map(normalize_text)
clean_df["response_clean"] = clean_df["response"].map(normalize_text)
clean_df["fingerprint"] = clean_df.apply(lambda row: fingerprint(row["prompt_clean"], row["response_clean"]), axis=1)
clean_df["prompt_tokens_clean"] = clean_df["prompt_clean"].map(token_estimate)
clean_df["response_tokens_clean"] = clean_df["response_clean"].map(token_estimate)

display(clean_df[["id", "source", "prompt_clean", "response_clean"]].head(8))

## Step 5: Drop structurally invalid rows

Empty responses and blank prompts should not survive into training data.

In [ ]:
structural_df = clean_df[
    (clean_df["prompt_clean"].str.len() > 0)
    & (clean_df["response_clean"].str.len() > 0)
].copy()

print("Rows after structural filter:", len(structural_df))
display(structural_df[["id", "task_type", "prompt_clean", "response_clean"]].head())

## Step 6: Remove exact duplicates

Exact duplicates artificially up-weight a pattern and can make a tiny dataset look larger than it really is.

In [ ]:
dedup_exact_df = structural_df.drop_duplicates(subset=["fingerprint"]).copy()

exact_removed = len(structural_df) - len(dedup_exact_df)
display(pd.DataFrame([{"stage": "exact_dedup", "removed_rows": exact_removed, "remaining_rows": len(dedup_exact_df)}]))

## Step 7: Remove near duplicates

Near duplicates are trickier because the text is not identical. Here we use a lightweight lexical Jaccard check that is simple enough to inspect and tune.

In [ ]:
kept_rows = []
near_duplicate_pairs = []

for row in dedup_exact_df.sort_values("id").to_dict("records"):
    duplicate_match = None
    for kept in kept_rows:
        same_task = row["task_type"] == kept["task_type"]
        prompt_sim = jaccard_similarity(row["prompt_clean"], kept["prompt_clean"])
        response_sim = jaccard_similarity(row["response_clean"], kept["response_clean"])
        if same_task and prompt_sim >= 0.82 and response_sim >= 0.75:
            duplicate_match = {
                "drop_id": row["id"],
                "keep_id": kept["id"],
                "prompt_similarity": round(prompt_sim, 3),
                "response_similarity": round(response_sim, 3),
            }
            break
    if duplicate_match:
        near_duplicate_pairs.append(duplicate_match)
    else:
        kept_rows.append(row)

dedup_df = pd.DataFrame(kept_rows)
near_dup_df = pd.DataFrame(near_duplicate_pairs)

display(near_dup_df if not near_dup_df.empty else pd.DataFrame({"message": ["No near duplicates flagged"]}))
print("Rows after exact + near dedup:", len(dedup_df))

## Step 8: Build a held-out evaluation reference set

Leakage checks compare candidate training examples against prompts we want to reserve for validation or later benchmark use.

In [ ]:
heldout_eval_prompts = [
    "What makes a useful held-out prompt for notebook-style evaluation?",
    "How should I compare a base model and a tuned model after a first run?",
    "How should a tutoring model respond when it is unsure about a detail?",
    "Explain when keeping an adapter separate is better than merging it.",
]

eval_reference_df = pd.DataFrame({"eval_prompt": heldout_eval_prompts})
display(eval_reference_df)

## Step 9: Flag likely train/eval leakage

For a small local pipeline, start with exact and near-text checks. More advanced semantic leakage checks can come later.

In [ ]:
leakage_rows = []

for row in dedup_df.to_dict("records"):
    best_overlap = 0.0
    closest_eval_prompt = None
    for eval_prompt in heldout_eval_prompts:
        overlap = jaccard_similarity(row["prompt_clean"], eval_prompt)
        if overlap > best_overlap:
            best_overlap = overlap
            closest_eval_prompt = eval_prompt
    leakage_rows.append(
        {
            "id": row["id"],
            "prompt_clean": row["prompt_clean"],
            "closest_eval_prompt": closest_eval_prompt,
            "max_overlap": round(best_overlap, 3),
            "leakage_flag": best_overlap >= 0.55,
        }
    )

leakage_df = pd.DataFrame(leakage_rows)
display(leakage_df.sort_values("max_overlap", ascending=False).head(8))

leak_free_ids = leakage_df.loc[~leakage_df["leakage_flag"], "id"]
leak_free_df = dedup_df[dedup_df["id"].isin(leak_free_ids)].copy()
print("Rows after leakage filter:", len(leak_free_df))

## Step 10: Shape a simple curriculum

Curriculum shaping is not about forcing one magic order. It is about making dataset difficulty visible so you can choose a deliberate mixture.

In [ ]:
difficulty_map = {"easy": 1, "medium": 2, "hard": 3}
task_weight_map = {"qa": 1.0, "rewrite": 1.1, "checklist": 1.15, "comparison": 1.25}

curriculum_df = leak_free_df.copy()
curriculum_df["difficulty_score"] = curriculum_df["difficulty"].map(difficulty_map)
curriculum_df["task_weight"] = curriculum_df["task_type"].map(task_weight_map)
curriculum_df["length_weight"] = curriculum_df["response_tokens_clean"].clip(lower=1) / 20
curriculum_df["curriculum_score"] = (
    curriculum_df["difficulty_score"] * 0.6
    + curriculum_df["task_weight"] * 0.7
    + curriculum_df["length_weight"] * 0.4
).round(2)
curriculum_df["curriculum_band"] = pd.cut(
    curriculum_df["curriculum_score"],
    bins=[0, 1.9, 2.6, 10],
    labels=["foundation", "core", "stretch"],
)

display(curriculum_df[["id", "task_type", "difficulty", "curriculum_score", "curriculum_band"]].sort_values("curriculum_score"))

## Step 11: Inspect class balance before sampling

Tiny datasets are especially sensitive to skew. If one task type dominates, the adapter may learn that style more strongly than you intended.

In [ ]:
balance_before_df = curriculum_df["task_type"].value_counts().rename_axis("task_type").reset_index(name="rows_before")
display(balance_before_df)

## Step 12: Balance the mixture

For this small course notebook we cap each task type at a similar count while preserving a spread of easy, medium, and hard examples.

In [ ]:
balanced_parts = []
for task_type, group_df in curriculum_df.sort_values(["curriculum_score", "id"]).groupby("task_type"):
    take_n = min(len(group_df), 3)
    balanced_parts.append(group_df.head(take_n))

balanced_df = pd.concat(balanced_parts, ignore_index=True).sort_values(["task_type", "curriculum_score", "id"]).reset_index(drop=True)
balance_after_df = balanced_df["task_type"].value_counts().rename_axis("task_type").reset_index(name="rows_after")

display(balance_after_df)
print("Balanced rows:", len(balanced_df))

## Step 13: Build a stage-by-stage comparison table

This table is one of the most useful artifacts in a curation notebook because it makes every cleaning decision visible.

In [ ]:
stage_counts_df = pd.DataFrame(
    [
        {"stage": "raw", "rows": len(raw_df)},
        {"stage": "structural_filter", "rows": len(structural_df)},
        {"stage": "exact_dedup", "rows": len(dedup_exact_df)},
        {"stage": "near_dedup", "rows": len(dedup_df)},
        {"stage": "leakage_filter", "rows": len(leak_free_df)},
        {"stage": "balanced_final", "rows": len(balanced_df)},
    ]
)

display(stage_counts_df)

ax = stage_counts_df.plot(x="stage", y="rows", kind="bar", legend=False, figsize=(8, 4), title="Rows remaining after each curation stage")
ax.set_ylabel("rows")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()

## Step 14: Design the validation split

We want the validation set to stay realistic and slightly challenging:

- keep at least one example from each task type
- include some medium and hard cases
- avoid obvious duplicates of training prompts

In [ ]:
validation_rows = []
training_rows = []

for task_type, group_df in balanced_df.groupby("task_type"):
    ordered = group_df.sort_values(["curriculum_score", "id"], ascending=[False, True]).reset_index(drop=True)
    validation_rows.append(ordered.iloc[0])
    if len(ordered) > 1:
        training_rows.extend(ordered.iloc[1:].to_dict("records"))

validation_df = pd.DataFrame(validation_rows).reset_index(drop=True)
training_df = pd.DataFrame(training_rows).reset_index(drop=True)

display(pd.DataFrame([
    {"split": "train", "rows": len(training_df)},
    {"split": "validation", "rows": len(validation_df)},
]))
display(validation_df[["id", "task_type", "difficulty", "prompt_clean"]])

## Step 14b: Audit split coverage

 A good split is not only the right size. It should also preserve enough task and difficulty diversity that validation still tests realistic behavior.

In [ ]:
split_audit_df = pd.concat(
    [
        training_df.assign(split="train"),
        validation_df.assign(split="validation"),
    ],
    ignore_index=True,
)

split_balance_df = (
    split_audit_df.groupby(["split", "task_type", "difficulty"])
    .size()
    .reset_index(name="rows")
    .sort_values(["split", "task_type", "difficulty"])
)

display(split_balance_df)

## Step 15: Prepare export-ready training records

Most training stacks want a compact record format. Here we keep both a generic prompt/response view and a chat-style record.

In [ ]:
def to_export_record(row):
    return {
        "id": row["id"],
        "task_type": row["task_type"],
        "difficulty": row["difficulty"],
        "prompt": row["prompt_clean"],
        "response": row["response_clean"],
        "curriculum_band": str(row["curriculum_band"]),
        "messages": [
            {"role": "system", "content": "You are Castalia Mentor, a concise tutor."},
            {"role": "user", "content": row["prompt_clean"]},
            {"role": "assistant", "content": row["response_clean"]},
        ],
    }

train_export_records = [to_export_record(row) for row in training_df.to_dict("records")]
validation_export_records = [to_export_record(row) for row in validation_df.to_dict("records")]

display(pd.DataFrame(train_export_records)[["id", "task_type", "difficulty", "curriculum_band"]].head())

## Step 16: Export JSONL and CSV artifacts

JSONL is convenient for training pipelines, while CSV is convenient for quick audit and spreadsheet inspection.

In [ ]:
train_jsonl_path = EXPORT_DIR / "train_curated.jsonl"
validation_jsonl_path = EXPORT_DIR / "validation_curated.jsonl"
train_csv_path = EXPORT_DIR / "train_curated.csv"
validation_csv_path = EXPORT_DIR / "validation_curated.csv"

for path_obj, records in [
    (train_jsonl_path, train_export_records),
    (validation_jsonl_path, validation_export_records),
]:
    with open(path_obj, "w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")

pd.DataFrame(training_df)[["id", "task_type", "difficulty", "prompt_clean", "response_clean"]].to_csv(train_csv_path, index=False)
pd.DataFrame(validation_df)[["id", "task_type", "difficulty", "prompt_clean", "response_clean"]].to_csv(validation_csv_path, index=False)

display(pd.DataFrame({"exported_file": [train_jsonl_path.name, validation_jsonl_path.name, train_csv_path.name, validation_csv_path.name]}))

## Step 17: Reload the exports to validate them

Never assume an export is correct just because the write step succeeded.

In [ ]:
reloaded_train_jsonl = [json.loads(line) for line in train_jsonl_path.read_text(encoding="utf-8").splitlines()]
reloaded_validation_jsonl = [json.loads(line) for line in validation_jsonl_path.read_text(encoding="utf-8").splitlines()]
reloaded_train_csv = pd.read_csv(train_csv_path)
reloaded_validation_csv = pd.read_csv(validation_csv_path)

validation_checks = [
    {"check": "train_jsonl_rows", "value": len(reloaded_train_jsonl), "expected": len(train_export_records)},
    {"check": "validation_jsonl_rows", "value": len(reloaded_validation_jsonl), "expected": len(validation_export_records)},
    {"check": "train_csv_rows", "value": len(reloaded_train_csv), "expected": len(training_df)},
    {"check": "validation_csv_rows", "value": len(reloaded_validation_csv), "expected": len(validation_df)},
]
validation_checks_df = pd.DataFrame(validation_checks)
validation_checks_df["pass"] = validation_checks_df["value"] == validation_checks_df["expected"]
display(validation_checks_df)

## Step 18: Save a short curation report

This turns notebook work into an artifact you can reuse in later training and evaluation notebooks.

In [ ]:
curation_report = {
    "raw_rows": int(len(raw_df)),
    "final_train_rows": int(len(training_df)),
    "final_validation_rows": int(len(validation_df)),
    "removed_exact_duplicates": int(exact_removed),
    "removed_near_duplicates": int(len(near_dup_df)),
    "leakage_flagged_rows": int(leakage_df["leakage_flag"].sum()),
    "task_balance_after": balance_after_df.to_dict("records"),
    "split_balance": split_balance_df.to_dict("records"),
}

report_path = ARTIFACT_DIR / "curation_report.json"
with open(report_path, "w", encoding="utf-8") as handle:
    json.dump(curation_report, handle, indent=2)

curation_report

## Key takeaways

- **Cleaning is measurable work**, not a vague preprocessing step.
- **Deduplication matters twice** on small datasets because duplicates distort both training and evaluation.
- **Leakage checks should happen before splitting**, not after the model already saw the data.
- **A lightweight curriculum and balanced mixture** often helps more than chasing exotic hyperparameters.